# Name : `Sai sourav Panigrahi`

## Project title: `Sign Language Gesture Recognition System using Machine Learning`

## problem Statement:
- Sign language is an essential means of communication for deaf and mute individuals. However, many people are unable to understand sign language, creating communication barriers in daily life. Manually interpreting sign language requires specialized knowledge and is not always available. Therefore, there is a need for an intelligent system that can automatically recognize sign language gestures from images and classify them accurately. This project aims to develop a machine learning-based image classification system that identifies sign language gestures and predicts their corresponding alphabet, number, or symbol.

## Objectives
### Primary Objective

To develop an image-based machine learning system that accurately recognizes and classifies sign language gestures.

### Specific Objectives
To collect and preprocess sign language gesture image data.
To prepare image data for machine learning model training.
To train and compare multiple machine learning algorithms for gesture classification.
To evaluate model performance using standard classification metrics.
To identify the best-performing model based on accuracy and other evaluation measures.
To deploy the final trained model using Streamlit for easy user interaction.

### Scope of the Project

The project focuses on recognizing static sign language gestures from images using machine learning techniques. It includes image preprocessing, model training, performance evaluation, and deployment through a Streamlit web application. The system is designed to classify gesture images into their respective sign language categories and provide accurate predictions. Real-time gesture recognition using a webcam and MediaPipe is considered as future work and is not included in the current implementation.

### Expected Outcome

The developed system should accurately classify sign language gesture images into their corresponding classes. The project will provide a user-friendly web interface where users can upload gesture images and receive predictions. The final solution is expected to demonstrate the practical application of machine learning in improving accessibility and communication.

------------------------------------

## Importing Required libraries

In [ ]:
# Basic Libraries
import os
import warnings
warnings.filterwarnings("ignore")

# Numerical & Data Handling
import numpy as np
import pandas as pd

# Image Processing
import cv2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Model Saving
import joblib

: 

# Dataset Loading

In [ ]:
import os

dataset_path = "dataset/Gesture Image Pre-Processed Data"

total_images = 0

for cls in os.listdir(dataset_path):
    total_images += len(os.listdir(os.path.join(dataset_path, cls)))

print("Total Images:", total_images)

### Observation

- The dataset contains 55,501 images.
- The dataset consists of 37 classes.
- The classes include digits (0–9), alphabets (A–Z), and one special symbol (_).

### Class Information

In [ ]:
classes = sorted(os.listdir(dataset_path))

print("Total Classes :", len(classes))
print(classes)

### Observation

- The dataset contains 37 gesture classes.
- Classes include digits (0–9), alphabets (A–Z), and one special symbol (_).
- These classes will be used as target labels for classification.

In [ ]:
images = []
labels = []

for label in classes:

    folder_path = os.path.join(dataset_path, label)

    for image_name in os.listdir(folder_path):

        image_path = os.path.join(folder_path, image_name)

        img = cv2.imread(image_path)

        if img is not None:
            images.append(img)
            labels.append(label)

print(len(images))
print(len(labels))

# Exploratory Data Analysis (EDA)

### Class Distribution Analysis

In [ ]:
class_counts = {}

for cls in classes:
    class_counts[cls] = len(
        os.listdir(os.path.join(dataset_path, cls))
    )

class_counts

### Observation

- Most classes contain approximately 1500 images.
- The dataset is balanced across all classes.
- No significant class imbalance is observed.

In [ ]:
df = pd.DataFrame(
    class_counts.items(),
    columns=["Class", "Image_Count"]
)

df.head()

In [ ]:
plt.figure(figsize=(15,5))

sns.barplot(
    data=df,
    x="Class",
    y="Image_Count"
)

plt.title("Images per Class")
plt.xticks(rotation=90)

plt.show()

### Observation

- The dataset shows a nearly uniform distribution across all gesture classes.
- Most classes contain approximately 1500 images.
- No significant class imbalance is present.
- The balanced dataset is suitable for machine learning classification.

### Image Dimension Analysis

In [ ]:
import cv2

sample_class = classes[0]

sample_image_path = os.path.join(
    dataset_path,
    sample_class,
    os.listdir(os.path.join(dataset_path, sample_class))[0]
)

img = cv2.imread(sample_image_path)

print("Image Shape :", img.shape)
print("Height :", img.shape[0])
print("Width :", img.shape[1])
print("Channels :", img.shape[2])

### Observation

- Image Shape: (50, 50, 3)
- Image Height: 50 pixels
- Image Width: 50 pixels
- Number of Channels: 3
- All images have fixed dimensions.

### Sample Gesture Images

In [ ]:
plt.figure(figsize=(12,8))

for i in range(9):
    idx = np.random.randint(0, len(images)-1)

    plt.subplot(3,3,i+1)
    plt.imshow(images[idx])
    plt.title(labels[idx])
    plt.axis('off')

plt.tight_layout()
plt.show()

### Observation

- Multiple gesture samples from different classes were visualized.
- The dataset contains a variety of hand gesture patterns representing digits, alphabets, and special symbols.
- All images have a consistent black background and white hand silhouette.
- The gesture shapes are clearly distinguishable, which can help machine learning models learn meaningful patterns.
- The dataset appears clean and well-preprocessed for classification tasks.

## Pixel Intensity Analysis

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(np.array(images).flatten(), bins=50)

plt.title("Pixel Intensity Distribution")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")

plt.show()

### Observation

- The pixel intensity distribution is concentrated around 0 and 255.
- Most pixels belong either to the black background or the white hand gesture region.
- The dataset exhibits high contrast between foreground and background.
- Clear separation of gesture and background can improve feature extraction and classification performance.
- The images are already well-preprocessed and suitable for machine learning training.

## Data Preprocessing

In [ ]:
## Convert to NumPy Array
X = np.array(images)
y = np.array(labels)

print("X Shape :", X.shape)
print("y Shape :", y.shape)

### Observation

- All images and corresponding labels were successfully loaded.
- The number of images and labels are equal.
- The dataset is ready for preprocessing.`m

### Observation

- The image dataset was successfully converted into NumPy arrays.
- X contains image data and y contains gesture labels.
- The dataset is ready for further preprocessing and model preparation.

In [ ]:
# Checking Pixel Range
print("Minimum Pixel Value :", X.min())
print("Maximum Pixel Value :", X.max())

## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

print("Total Classes :", len(le.classes_))
print("First 10 Classes :", le.classes_[:10])

### Observation

- Label Encoding successfully converted gesture labels into numerical values.
- The dataset contains 37 unique gesture classes.
- Digits (0–9), alphabets (A–Z), and special symbol (_) are included.
- Encoded labels are ready for machine learning model training.

## Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("X_train Shape :", X_train.shape)
print("X_test Shape :", X_test.shape)
print("y_train Shape :", y_train.shape)
print("y_test Shape :", y_test.shape)

### Observation

- The dataset was successfully split into training and testing sets.
- 80% of the data (44,400 images) was used for training.
- 20% of the data (11,100 images) was reserved for testing.
- Stratified sampling was applied to preserve class distribution across all 37 gesture classes.
- The training and testing datasets are ready for preprocessing and feature extraction.

## Image Normalization

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

print("Train Min :", X_train.min())
print("Train Max :", X_train.max())

print("Test Min :", X_test.min())
print("Test Max :", X_test.max())

### Observation

- Pixel values were successfully normalized from the range 0–255 to 0–1.
- Normalization helps improve model learning and numerical stability.
- Both training and testing datasets now have a consistent scale.
- The normalized images are ready for feature extraction.

## Feature Extraction using HOG

In [ ]:
from skimage.feature import hog
from skimage.color import rgb2gray
import numpy as np

X_train_hog = []
X_test_hog = []

# Training Images
for image in X_train:

    gray = rgb2gray(image)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    X_train_hog.append(features)

# Testing Images
for image in X_test:

    gray = rgb2gray(image)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    X_test_hog.append(features)

X_train_hog = np.array(X_train_hog)
X_test_hog = np.array(X_test_hog)

print("X_train_hog Shape :", X_train_hog.shape)
print("X_test_hog Shape :", X_test_hog.shape)

### Observation

- HOG (Histogram of Oriented Gradients) features were successfully extracted from all gesture images.
- Each image was converted into a feature vector containing 900 shape-based features.
- HOG captures edge, contour, and hand shape information, which is highly relevant for sign language recognition.
- The original image dimensions (50×50×3) were transformed into compact feature vectors suitable for machine learning algorithms.
- The extracted features are ready for feature scaling and model training.

## Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_hog)
X_test_scaled = scaler.transform(X_test_hog)

print("X_train_scaled Shape :", X_train_scaled.shape)
print("X_test_scaled Shape :", X_test_scaled.shape)

### Observation

- HOG feature vectors were successfully standardized using StandardScaler.
- The scaler was fitted only on the training data to prevent data leakage.
- Both training and testing feature sets now have a consistent scale.
- Feature scaling improves the performance of distance-based and optimization-based machine learning algorithms such as KNN and SVM.
- The scaled feature matrix is ready for model training.

# Model Building

## K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(
    n_neighbors=5
)

knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

knn_accuracy = accuracy_score(y_test, y_pred_knn)

print("KNN Accuracy :", round(knn_accuracy * 100, 2), "%")

### Observation

- The K-Nearest Neighbors (KNN) classifier was successfully trained using the scaled HOG feature vectors.
- The model achieved an accuracy of 99.36% on the test dataset.
- The high accuracy indicates that HOG features effectively capture the shape and edge patterns of hand gestures.
- KNN was able to accurately distinguish between the 37 gesture classes present in the dataset.m
- The model demonstrated excellent classification performance and serves as a strong benchmark for comparison with other machine learning algorithms.

## Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

gnb = GaussianNB()

gnb.fit(X_train_scaled, y_train)

y_pred_gnb = gnb.predict(X_test_scaled)

gnb_accuracy = accuracy_score(y_test, y_pred_gnb)

print("Gaussian Naive Bayes Accuracy :", round(gnb_accuracy * 100, 2), "%")

### Observation

- The Gaussian Naive Bayes model was trained on the extracted HOG features.
- The model achieved an accuracy of 87.78% on the test dataset.
- Although the model performed reasonably well, its accuracy was lower than KNN.
- This is because Naive Bayes assumes feature independence, while HOG features are often correlated.
- The model serves as a useful baseline for comparison with more advanced machine learning algorithms.

## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train_scaled, y_train)

y_pred_dt = dt.predict(X_test_scaled)

dt_accuracy = accuracy_score(y_test, y_pred_dt)

print("Decision Tree Accuracy :", round(dt_accuracy * 100, 2), "%")

### Observation

- The Decision Tree classifier was successfully trained using the scaled HOG feature vectors.
- The model achieved an accuracy of 97.14% on the test dataset.
- The model was able to learn important decision rules from the extracted gesture features.
- Decision Tree performed significantly better than Gaussian Naive Bayes and demonstrated strong classification capability.
- However, its accuracy was slightly lower than KNN, indicating that some gesture patterns may require more robust generalization.

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)

rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy :", round(rf_accuracy * 100, 2), "%")

### Observation

- The Random Forest classifier was successfully trained using the scaled HOG feature vectors.
- The model achieved an accuracy of 99.98% on the test dataset.
- Random Forest significantly improved the classification performance by combining predictions from multiple decision trees.
- The ensemble learning approach reduced overfitting and enhanced generalization capability.
- The model demonstrated exceptional performance in recognizing sign language gestures and achieved the highest accuracy among all tested models.

# Model Evaluation
## Accuracy Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": [
        "KNN",
        "Gaussian Naive Bayes",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy (%)": [
        99.36,
        87.78,
        97.14,
        99.98
    ]
})

results = results.sort_values(
    by="Accuracy (%)",
    ascending=False
)

results

## Accuracy Comparison Visualization

In [ ]:
plt.figure(figsize=(8,5))

ax = sns.barplot(
    data=results,
    x="Model",
    y="Accuracy (%)",
    palette="Set2",
    hue="Model",
    legend=False
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f")

plt.title("Model Accuracy Comparison", fontsize=14)
plt.xlabel("Models")
plt.ylabel("Accuracy (%)")
plt.ylim(80, 101)

plt.tight_layout()
plt.show()

### Observation

- The accuracy comparison chart visually illustrates the performance of all evaluated machine learning models.
- Random Forest achieved the highest accuracy of 99.98%, making it the best-performing model.
- KNN achieved an accuracy of 99.36% and demonstrated excellent classification performance.
- Decision Tree achieved 97.14% accuracy and produced strong classification results.
- Gaussian Naive Bayes achieved the lowest accuracy of 87.78% among the evaluated models.
- Based on the comparison, Random Forest was selected as the final model for deployment.

## Best Model Selection

In [ ]:
best_model = results.iloc[0]

print("Best Model :", best_model["Model"])
print("Accuracy :", best_model["Accuracy (%)"], "%")

## Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_rf))

### Observation

- The classification report was generated using the Random Forest model.
- The model achieved excellent precision, recall, and F1-score values across all gesture classes.
- Most classes obtained a score of 1.00, indicating highly accurate classification performance.
- The overall accuracy of the model was approximately 100% on the test dataset.
- The balanced dataset and HOG feature extraction contributed significantly to the model's outstanding performance.
- The results confirm that the Random Forest model is highly effective for sign language gesture recognition.

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(12,12))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm
)

disp.plot(
    cmap="Blues",
    ax=ax,
    include_values=False
)

plt.title("Confusion Matrix - Random Forest")
plt.show()

### Observation

- The confusion matrix was generated for the Random Forest model.
- Most predictions are concentrated along the main diagonal, indicating correct classifications.
- Very few misclassifications were observed among the 37 gesture classes.
- The matrix demonstrates the excellent classification capability of the Random Forest model.
- The results are consistent with the achieved accuracy of 99.98%.

# Model Saving

In [ ]:
import joblib

joblib.dump(rf, "sign_language_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")